# B5 — Purpose Classification

Assigns purposes to trajectories based on **dominant era** of meaningful+ interactions.
Category-based purpose was useless (50/51 FineArtsPurpose due to 93% Painting).
Era-based purpose uses `poi:creationPeriod` data (already in KG) to distinguish:
- **ClassicalArtsPurpose**: dominant era is Pre-1600, 17th, or 18th century
- **ModernArtsPurpose**: dominant era is 19th century
- **ContemporaryArtsPurpose**: dominant era is Early or Late 20th century+
- **GeneralPurpose**: no era exceeds 50% of dwell time

In [ ]:
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.abspath('helpers'))
from fuseki_config import query_endpoint, update_endpoint
from fuseki_queries import execute_sparql_query, execute_sparql_update, format_results
from artwork_categories import ERA_TO_PURPOSE, resolve_era

In [ ]:
PREFIXES = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi:      <http://w3id.org/trajecto/poi#>
PREFIX purpose:  <http://w3id.org/trajecto/purpose#>
PREFIX dul:      <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX xsd:      <http://www.w3.org/2001/XMLSchema#>
"""

## 1. Get Era Engagement per Trajectory

In [ ]:
query = PREFIXES + """
SELECT ?traj ?poi ?dwell ?period
WHERE {
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:hasInteraction ?interaction .
    ?interaction poi:interactsWith ?poi .
    ?interaction poi:hasDwellTimeInSeconds ?dwell .
    ?interaction a poi:MeaningfulInteraction .
    ?poi poi:creationPeriod ?period .
}
"""
result = execute_sparql_query(query_endpoint, query)
df = format_results(result)
df['dwell'] = df['dwell'].astype(float)
print(f'Trajectory-artwork pairs with era data: {len(df)}')

# Resolve eras
df['era'] = df['period'].apply(resolve_era)
print('\nEra distribution:')
print(df['era'].value_counts())

## 2. Determine Dominant Era per Trajectory

In [ ]:
# Aggregate dwell by era per trajectory
era_agg = df.groupby(['traj', 'era']).agg(
    totalDwell=('dwell', 'sum'),
    count=('dwell', 'count')
).reset_index()

# Remove Unknown era from dominance calculation
era_agg = era_agg[era_agg['era'] != 'Unknown']

purposes = []
for traj, group in era_agg.groupby('traj'):
    total_dwell = group['totalDwell'].sum()
    dominant = group.loc[group['totalDwell'].idxmax()]
    dominant_ratio = dominant['totalDwell'] / total_dwell if total_dwell > 0 else 0

    if dominant_ratio > 0.5:
        purpose = ERA_TO_PURPOSE.get(dominant['era'], 'GeneralPurpose')
    else:
        purpose = 'GeneralPurpose'

    purposes.append({
        'traj': traj,
        'purpose': purpose,
        'dominant_era': dominant['era'],
        'dominant_ratio': dominant_ratio,
    })

purposes_df = pd.DataFrame(purposes)
print('Purpose distribution:')
print(purposes_df['purpose'].value_counts())
print('\nDominant era distribution:')
print(purposes_df['dominant_era'].value_counts())
print('\nDominant ratio stats:')
print(purposes_df['dominant_ratio'].describe())

## 3. Insert Purposes into KG

In [ ]:
insert_lines = []
for _, row in purposes_df.iterrows():
    insert_lines.append(
        f'  <{row["traj"]}> trajecto:hasPurpose purpose:{row["purpose"]} .'
    )

if insert_lines:
    triples = '\n'.join(insert_lines)
    update = PREFIXES + f"""INSERT DATA {{\n{triples}\n}}"""
    success = execute_sparql_update(update_endpoint, update)
    print(f'Inserted {len(insert_lines)} purpose assignments: {"OK" if success else "FAILED"}')

## 4. Verify

In [ ]:
q = PREFIXES + """
SELECT ?purpose (COUNT(?traj) AS ?count)
WHERE {
    ?traj trajecto:hasPurpose ?purpose .
}
GROUP BY ?purpose
ORDER BY DESC(?count)
"""
result = execute_sparql_query(query_endpoint, q)
format_results(result)